# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 0. 제출 정보
- 이름: 김지인
- GitHub ID: jiin-0
- 작성일: 2026.10.07
- 최종 제출 URL: https://github.com/jiin-0/llm-data-analysis-study/blob/main/chapter04/chapter04_pandas_basic.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
 상태가 "completed"인 주문의 총 금액은 여성이 더 클까 남성이 더 클까

### 사용한 파일/컬럼
 customers-gender/customer_id

 orders-customer_id/order_id/order_status

 order_items-order_id/quantity/unit_price


### 결과 관찰
[분석을 마치고 결과를 관찰하라는 의미가 아닌, 파일과 컬럼 선정에 대한 결과를 작성하라고 이해했습니다.]

customers에는 gender(성별)와 customer_id에 해당하는 컬럼이 존재한다. customers파일을 통해 주문자의 성별을 알 수 있다.
orders파일에는 customer_id, order_id, order_status 컬럼이 존재한다. 따라서 이를 통해 'completed' 주문을 선별할 수 있고, 이를 customers와 order_items와 연결할 수 있다.
order_items에는 order_id, quantity, unit_price 컬럼이 존재한다. 이를 통해 각 주문별 주문 총액을 계산할 수 있다.

### 나의 해석과 판단
질문에 답을 하기 위해서는 성별과 고객을 연결할 정보, 고객의 주문정보, 고객의 주문 총액, 주문의 상태를 알아야한다. 따라서 각 정보가 포함되어 있는 컬럼(customers의 customer_id, gender/orders의 order_id, order_status/order_items의 quantity, unit_price)들을 선정하였고, 각 정보를 유기적으로 연결해줄수 있는 컬럼(orders의 customer_id/order_items의 order_id)들을 선정하였다. 따라서 필요한 정보를 정리하고 공통컬럼을 통해 정보들을 종합해 성별에 따른 완료 주문 총액을 계산할 수 있을 것이라고 생각했다.

### 업무·분석적 의미
필요한 파일과 컬럼을 정리하여 어떤 파일에 어떤 컬럼이 있는지 파악이 필요하고, 또한 필요한 컬럼이 어떤 것인지 파악하여 데이터 분석의 방향성을 결정할 수 있다.

### 한계와 추가 확인 사항
각 공통 컬럼으로 연결할 때 오류가 발생할 수 있다. 따라서 계획을 위처럼 수립하나, 실제 데이터 값중 null이 존재할 수 있으므로 데이터 분석 시 단계별로 데이터의 상황을 확인해야 한다.

![필터와 파생 컬럼](images/step01_question.png)

## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [7]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-study\notebooks\ch04
프로젝트 루트: C:\dev\llm-data-analysis-study
데이터 폴더: C:\dev\llm-data-analysis-study\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-study\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [8]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [38]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


In [41]:
# [과제용]

customer_gender=customers[['customer_id','gender']]
display(customer_gender.head())

print('성별별 고객 수: ', customer_gender['gender'].value_counts())

orders_gender=orders[['customer_id','order_id','order_status']]
display(orders_gender.head())

order_items_gender=order_items[['order_id','quantity','unit_price']]
display(order_items_gender.head())



,customer_id,gender
0,1,F
1,2,F
2,3,F
3,4,F
4,5,F


성별별 고객 수:  gender
F    84
M    66
Name: count, dtype: int64


,customer_id,order_id,order_status
0,123,1,completed
1,77,2,cancelled
2,138,3,cancelled
3,57,4,cancelled
4,125,5,cancelled


,order_id,quantity,unit_price
0,1,3,102000
1,1,5,25000
2,1,3,142000
3,1,3,193000
4,2,4,189000


## 5. 정렬과 주문 상태 확인


In [39]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-10
8,9,송지민,M,69,서울,2025-12-25
54,55,윤영철,M,69,광주,2023-10-06
78,79,이서준,M,69,부산,2023-09-19
136,137,김선영,M,68,울산,2025-02-03
45,46,김서현,M,67,대전,2026-07-19
132,133,김성진,M,67,성남,2026-08-04
123,124,김시우,M,67,대구,2024-11-30
11,12,김정남,F,66,대전,2024-12-06
57,58,김성훈,F,66,성남,2025-08-20


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [42]:
# [과제용]
print('주문 상태별 건수')
display(orders_gender['order_status'].value_counts())

주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [11]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


In [43]:
# [과제용]
order_items_gender['line_total']=order_items_gender['quantity']*order_items_gender['unit_price']

display(order_items_gender.head())

,order_id,quantity,unit_price,line_total
0,1,3,102000,306000
1,1,5,25000,125000
2,1,3,142000,426000
3,1,3,193000,579000
4,2,4,189000,756000


## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: 
    예시 코드: 30세 이상, 도시가 부산 혹은 서울인 고객

- 정렬 기준:
- 예시 코드: 높은 가격순, 고령순, 주문 상태별
- 과제용(작성) 코드: 성별

- 만든 파생 컬럼: line_total
- `line_total` 계산식: order_items['quantity']*order_items['unit_price']

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰-예시 코드 기준(예시 코드가 더 많은 항목을 포함)

성남, 광주, 부산, 대구, 서울, 울산, 인천, 대전, 수원, 고양의 10개 도시가 관찰되었다. 또한, 30세 이상 고객은 총 111명으로 나타났으며, 도시가 서울인 고객은 15명, 서울 또는 부산인 고객은 31명으로 나타났다.

높은 가격순으로 정렬한 결과 가장 고가의 상품은 200000, 고령순으로 정렬한 결과 가장 고령의 고객은 69세였다. completed된 주문은 184건, cancelled 주문은 64건, refunded 주문은 52건으로 관찰되었다.

quantity와 unit_price를 곱해 line_total을 생성했고, 각 주문 상세 행별 금액이 계산되었다.
전체 주문 상세 금액은 255610000로 나타났다.


### 나의 해석과 판단
필터의 기준이 달라지면 산출물이 달라질 것이다. 또한, 원하는 데이터를 얻기 위해서는 필터 조건을 잘 설정해야한다. 예를 들어 현재의 전체 주문 상세 금액에는 주문 상태와 관련된 필터가 걸려있지 않다. 따라서 실제 수익과는 차이가 있을 수 있다.


### 업무·분석적 의미
필터를 설정하여 불필요한 정보를 제거해 필요한 정보만 볼 수 있게 데이터의 양을 줄일 수 있다. 또한, 적절한 필터를 걸어야 원하는 데이터를 선별할 수 있다. 데이터 간 연산 시 불필요한 데이터가 포함되어 원하는 결과와 다른 연산이 작동될 수 있어 유의해야한다.

### 한계와 추가 확인 사항
현재 단순히 unit_price가 숫자이므로 화폐 단위를 알 수 없다. 따라서 데이터 생성 소스와 관련하여 화폐 단위를 확인해야하며, 전체 주문 상세 금액에 completed 주문, cancelled 주문, refunded 주문이 모두 포함되어 있으므로 목적에 따라 계산 항목에 추가 필터를 걸어야 한다.

더불어 현재의 나이는 기준이 불분명한 나이이다. 데이터 생성 시점이 기준일 수도 있고, 가입일이 기준일 수 있다. 따라서 나이 데이터의 시점을 확인하여 필요할 경우 수정이 필요하다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [51]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [54]:
# [과제용]
print('orders_gender.order_id 중복 수:', orders_gender['order_id'].duplicated().sum())

order_gender=order_items_gender.merge(
    orders_gender[['order_id','customer_id','order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print(order_gender.head())
print('병합 전 행 수:', len(order_items_gender))
print('병합 후 행 수:', len(order_gender))
display(order_gender['_merge'].value_counts())
order_gender=order_gender.drop(columns='_merge')

print(order_gender.head())


orders_gender.order_id 중복 수: 0
   order_id  quantity  unit_price  line_total  customer_id order_status _merge
0         1         3      102000      306000          123    completed   both
1         1         5       25000      125000          123    completed   both
2         1         3      142000      426000          123    completed   both
3         1         3      193000      579000          123    completed   both
4         2         4      189000      756000           77    cancelled   both
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

   order_id  quantity  unit_price  line_total  customer_id order_status
0         1         3      102000      306000          123    completed
1         1         5       25000      125000          123    completed
2         1         3      142000      426000          123    completed
3         1         3      193000      579000          123    completed
4         2         4      189000      756000           77    cancelled


## 3. merge 검증-작성 코드 기준
- 병합한 데이터: orders_gender, order_items_gender
- 사용한 key: order_id
- `validate` 결과: many_to_one으로 설정하였고, 병합 전후로 order_items의 행의 수와 전체 행이 동일하므로 validate가 제대로 작동하였다.
- `indicator` 결과: indicator 결과 both가 764(전체), left_only와 right_only가 0개이므로 order_items_gender의 모든 행이 orders_gender와 매치되었다.
- 병합 전/후 행 수: 764->764

![merge 검증](images/step03_merge.png)

### 결과 관찰
order_id를 기준으로 merge 되었다. 모든 order_items_gender의 행들이 orders_gender의 데이터와 매치되어 customer_id, order_status column을 갖게 되었다.

### 나의 해석과 판단
이번 병합은 모든 데이터가 매치되었으므로 안전하다고 판단했다. orders_gender의 order_id는 중복없이 고유하며, order_items_gender와의 매치에서 행 수가 줄거나 감소하지 않았으므로, items의 데이터가 orders에 매치되었다고 판단하였다.

만약 merge이후 행수가 감소한 경우, 이는 left_only 행이 삭제된 경우로, order_items_gender의 특정 행의 order_id와 같은 값을 갖는 행이 orders_gender에 존재하지 않는다는 뜻이다. 반대로 merge 후 행이 증가한 경우, 이는 order_items_geder의 특정 행이 orders_gender의 여러 행에 중복 매치된 경우를 의미한다. 이번 병합의 경우, orders_gender의 order_id 값이 고유한 것을 확인하였으므로, 행이 증가하는 경우는 발생하지 않았다.

### 업무·분석적 의미
잘못 merge할 경우, 데이터가 중복될 수 있고, 또한 필요한 데이터가 삭제될 수 있다. 따라서 merge의 결과를 제대로 확인하여 오류가 발생하지 않았는지 점검해야 한다. 확인하지 않고, 잘못 병합된 상태에서 데이터 분석을 진행할 경우, 데이터 분석 코드 실행에서 오류가 발생할 수 있다. 또한 결합 기준(Key)을 잘못 설정하면 왜곡으로 인해 잘못된 통계치나 결론을 도출할 위험이 크다. 이는 최종적으로 데이터 분석 전체의 신뢰성을 떨어뜨리고, 잘못된 의사결정으로 이어질 수 있으므로 분석 전 결과 검증(Validate) 작업이 필수적이다.

### 한계와 추가 확인 사항
동일한 의미의 키일지라도 데이터 타입이나 문자열 공백, 대소문자 차이 등으로 인해 정상적으로 매칭되지 않고 손실될 수 있다. 또한, 행의 수가 일치하는 것은 확인했지만, 결측이 존재하는지 확인하지 않았으므로, 결측을 확인하면 정확성을 올릴 수 있다.


## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [57]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


In [58]:
# [과제용]
completed_order_gender=order_gender[
    order_gender['order_status']=='completed'
].copy()

print(completed_order_gender.head())
print(completed_order_gender['order_status'].value_counts())

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_gender['line_total'].sum())

    order_id  quantity  unit_price  line_total  customer_id order_status
0          1         3      102000      306000          123    completed
1          1         5       25000      125000          123    completed
2          1         3      142000      426000          123    completed
3          1         3      193000      579000          123    completed
12         6         3       24000       72000           87    completed
order_status
completed    474
Name: count, dtype: int64
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [60]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [15]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


In [ ]:
# [과제용]

completed_order_gender_groupby=completed_order_gender.groupby(["order_id", "customer_id", "order_status"], as_index=False).agg(total_price=("line_total", "sum"))

print(completed_order_gender_groupby.head())

customer_gender.customer_id 중복 수: 0
   order_id  customer_id order_status  total_price
0         1          123    completed      1436000
1         6           87    completed      1309000
2         9          145    completed       236000
3        11           47    completed       222000
4        12           98    completed       444000


In [65]:
# [과제용]
print('customer_gender.customer_id 중복 수:', customer_gender['customer_id'].duplicated().sum()) 

completed_gender=completed_order_gender_groupby.merge(
    customer_gender[['customer_id', 'gender']],
    on='customer_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print(completed_gender.head())
print('병합 전 행 수:', len(completed_order_gender_groupby))
print('병합 후 행 수:', len(completed_gender))
display(completed_gender['_merge'].value_counts())
completed_gender=completed_gender.drop(columns='_merge')

print(completed_gender.head())

customer_gender.customer_id 중복 수: 0
   order_id  customer_id order_status  total_price gender _merge
0         1          123    completed      1436000      M   both
1         6           87    completed      1309000      M   both
2         9          145    completed       236000      M   both
3        11           47    completed       222000      F   both
4        12           98    completed       444000      F   both
병합 전 행 수: 184
병합 후 행 수: 184


_merge
both          184
left_only       0
right_only      0
Name: count, dtype: int64

   order_id  customer_id order_status  total_price gender
0         1          123    completed      1436000      M
1         6           87    completed      1309000      M
2         9          145    completed       236000      M
3        11           47    completed       222000      F
4        12           98    completed       444000      F


In [ ]:
# [과제용]
completed_gender_2=completed_gender.groupby(["customer_id", "gender"], as_index=False).agg(total_price=("total_price", "sum"))

print(completed_gender_2.head())

gender_total_price=completed_gender_2.groupby('gender')[['total_price']].sum()

print(gender_total_price.head())

print("결측 수:", gender_total_price.isna().sum())
# print(gender_total_price.isna().sum().sum() > 0)

   customer_id gender  total_price
0            3      F      3178000
1            4      F       603000
2            5      F      2004000
3            6      F      1349000
4            7      F      1173000
        total_price
gender             
F          77708000
M          71282000
결측 수: total_price    0
dtype: int64
False


## 11. 월별 매출


In [16]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,22314000,24,929750.0
4,2026-01,13303000,16,831438.0
5,2026-02,5318000,7,759714.0
6,2026-03,17529000,24,730375.0
7,2026-04,11832000,17,696000.0
8,2026-05,16345000,18,908056.0
9,2026-06,14070000,18,781667.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [17]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 4. completed 주문 범위와 집계
- 분석 범위 정의: order_sales에서 order_status가 completed인 주문
- 카테고리별 결과: category를 기준으로 groupby하여 총 주문 수와 판매금액을 계산하고, 전체 판매액 중 차지하는 비율을 구함.
- 상품별 결과: product_id를 기준으로 groupby하여 상품별 총 주문 금액과 주문 수를 계산.
- 월별 결과: order_month를 기준으로 groupby하여 월별 총 주문 금액과 주문 수를 계산.
- 고객별 결과: customer_id를 기준(+city)으로 groupby하여 고객별 총 주문금액과 주문 수를 계산.

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
- completed 주문을 집계한 결과 총 764개 행 중 474개의 주문 상품 행이 남았다.
- 카테고리별 매출 집계 결과, 스포츠에서 가장 큰 total_sales가 나타남. 전체의 21.31%, 패션이 가장 낮은 판매액을 보였고, 7.11%로 나타났다.
- 상품별 매출 집계 결과, 스포츠 상품041이 가장 높은 주문 금액을 보였고, 상위 10개 중 3개가 스포츠 상품, 2개가 생활용품 상품, 전자기기가 2개, 나머지가 각각 식품, 뷰티, 패션으로 나타났다.
- 월별 매출 집계 결과, 2025-09부터 2026-09로 데이터가 나타났으며, 2025-12와 2026-03에 24건으로 가장 많은 주문이 이루어졌고, 평균 금액은 2026-09가 가장 높았다.
- 고객별 구매 금액 집계 결과, customer 117이 가장 큰 금액을 구매했고, 상위 10명의 고객 중 서울이 3명, 성남이 2명, 나머지가 각각 고양, 수원, 인천, 광주, 부산으로 나타났다.

### 나의 해석과 판단
월별 매출 집계 결과가 가장 중요하다고 생각했다. 분석 결과, 월별 주문 수에서는 최솟값이 3, 최댓값이 24로 편차가 크게 나타났지만, 평균 주문 금액은 2025-09를 제외하면 대체적으로 유사하게 나타났다. 평균주문 금액이 일정하므로 전체 주문 금액은 주문수에 크게 영향을 받고 있다고 생각했다.

### 업무·분석적 의미
판매액이 주문수에 크게 영향을 받으므로, 각 상품의 객단가 상승보다는 마케팅을 활용해 주문건수를 높이는 방향으로 고객들을 노려야할 것이다.
또한, 주문 건수가 월별 차이가 크기 때문에 비수기(2월, 8월)에는 마케팅에 더 신경 쓸 필요가 있다.

### 한계와 추가 확인 사항
현재 데이터의 시작 시점이 25-09-14, 끝 시점이 26-09-13이므로, 2025-09와 2026-09는 주문건수가 전체 한 달을 대표하지 못한다. 따라서 필요한 경우 두 값을 더해 생각하거나 이후 데이터를 추가적으로 확보하는 절차가 필요하다.

이번 분석으로 계산한 `total_sales`는 회계상 순매출과 같다고 단정할 수 없다. 그 까닭은, 현재의 total_sales는 단순히 상품의 가격*판매 수이므로, 할인한 가격이나 쿠폰으로 인해 실제 감소된 지불이 반영되지 않았기 때문이다. 따라서 현재의 데이터 상에서는 주문건별 평균 금액이 유사하게 나타나나 특정 시기에 할인쿠폰이나 세일 이벤트를 진행했을 경우 주문 평균 금액상의 차이가 더 커질 수 있어 유의해야한다.

In [94]:
print("원본 합계: ", completed_order_sales['line_total'].sum())
print("카테고리 합계: ", category_sales['total_sales'].sum())
print("월별 합계: ", monthly_summary['total_sales'].sum())
print("고객별 합계: ", customer_sales['total_sales'].sum())

원본 합계:  148990000
카테고리 합계:  148990000
월별 합계:  148990000
고객별 합계:  148990000


## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계: 148990000
- 차이 여부: X

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
총합 검증을 하지 않을 경우, 전처리나 병합 단계에서 발생한 데이터 중복 생성 및 누락 건을 인지하지 못한 채 후속 분석을 진행할 수 있어 확인이 필요하다.

 또한, 총합 검증결과 총 금액 불일치가 일어난 경우, completed 필터가 동일하게 적용되었는지, line_total 계산식이 동일한지 오류가 없는지 확인해야한다. 이후 merge 과정에서 행이 증가하거나 감소하지 않았는지 확인해야하며, groupby에서 결측 key가 존재하지 않는지 확인하고, 일부 데이터만 다시 필터링 하지 않았는지 점검해야한다. 또한 데이터 형식이 숫자열인지 확인해야한다.

## 13. 결과 저장하기


In [18]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


In [80]:
# [과제용]
output={
    'ch04_gender_total_price.csv': gender_total_price
}
for filename, df in output.items():
    path=REPORT_DIR/filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)

ch04_gender_total_price.csv True 36


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해 안전한 수정 코드와 검증 순서를 설명해 주세요.
```


[LLM 코드 검증]

원본의 order_items에는 category 열이 존재하지 않고, 'line_total' 열이 존재하지 않는다. 또한, 주문 상태가 completed, cancelled, refunded로 나뉘어져 있어 completed만 고르는 절차가 필요하다.

category 열이 존재하는 products와 merge가 필요하고, line_total을 계산하는 수식을 추가해 열을 덧붙여야 한다. 또한, order_status가 존재하는 orders와 merge가 필요하고, order_status=completed인 행만 남겨야한다.

아래의 코드를 참고해주세요.


In [95]:
# 전처리 되었다고 가정
order_items = order_items.copy() #원본데이터 손실을 막기 위한 copy
order_items['line_total'] = order_items['quantity'] * order_items['unit_price'] #line_total 계산

print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum()) #merge를 위해 key 중복여부 확인 (확인 1)

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)  #merge-items와 orders

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())
#병합 결과 손실 확인(확인 2)

order_sales = order_sales.drop(columns='_merge') #_merge column 삭제(불필요)

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()
#order_status가 완료인 주문만 남기기


print('products.product_id 중복 수:', products['product_id'].duplicated().sum()) #products와 merge를 위해 key 중복 확인 (확인 3)

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
) #merge

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())
#merge 전후 손실 확인(확인 4)

completed_sales_items = completed_sales_items.drop(columns='_merge')
#불필요한 column제거

category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
#category를 기준으로 groupby 후 total_sales, total_quantity 계산

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

print("원본 합계: ", completed_order_sales['line_total'].sum())
print("카테고리 합계: ", category_sales['total_sales'].sum())

#총액 손실 확인 (확인 4)

orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


원본 합계:  148990000
카테고리 합계:  148990000


## 6. LLM pandas 코드 검증
- LLM Prompt 요약:
- customers와 orders, order_items를 사용해 고객별 구매 금액을 계산하려고 함.
다음 조건을 포함해 python 코드를 짤 것.
1. quantity*unit_price로 line_total 생성
2. orders.order_id, customer_id의 고유성 확인
3. many_to_one merge와 indicator 사용
4. 병합 전후 행 수와 비매칭 확인
5. order_status==completed만 필터링
6. 원본 completed total과 고객별 total 합계를 비교


- 제안 코드 요약:
  주문 상품별 구매 금액(quantity × unit_price)을 계산한 뒤, 주문 정보 및 고객 정보와 병합하여 고객별 구매 금액을 구한다. 병합 전후의 행 수와 매칭되지 않은 데이터를 확인하고, 주문 번호와 고객 번호의 고유성도 점검한다. 이후 order_status가 completed인 주문만 남겨 고객별 금액을 합산하고, 그 합계가 completed 주문 상품 금액의 전체 합계와 일치하는지 비교한다. 구매 내역이 없는 고객은 금액을 0으로 표시한다.(제안코드 전문은 하단에 code로 기록)


- 실제 컬럼/범위와 맞지 않은 부분: 없다. 필요한 컬럼을 적절히 사용하였다. 프롬프트상 언급된 'unit_price', 'quantity'를 사용해 'line_total'을 올바르게 계산하였고, orders.order_status로 completed를 적절히 필터링했고, merge도 문제없이 이루어졌다. 또한, 주문이 없는 고객의 데이터는 0으로 채워 결측이 발생하지 않도록 대처하였다.

- 수정한 내용: 코드의 오류는 없으나 최종 결과에 '이름'이라는 개인정보가 포함된다. 따라서 'name' 컬럼을 제거한 뒤 사용한다.(drop(columns='name'))
  또한 주문이 없는 고객의 값을 0으로 채워넣는 추가적인 절차를 거쳤다. 따라서 주문이 없는 고객의 칸을 채우지 않는다면 코드 마지막에 추천된 customer_purchase 코드부터는 삭제하여도 무방하다.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
단순히 코드가 실행되었다는 것이 분석이 맞다는 것을 의미하지 않기 때문이다. 단순하게 print("Hello world")라는 코드는 문제없게 실행되겠지만, 원하는 데이터 분석이 되었다고는 볼 수 없기 때문이다. 따라서 코드에서 처리되는 내용들이 나의 분석의 목적에 부합한지 확인이 필요하다.

In [114]:
import numpy as np
import pandas as pd

# 키의 결측값과 고유성 확인
for df_name, df, key in [
    ("orders", orders, "order_id"),
    ("orders", orders, "customer_id"),
    ("order_items", order_items, "order_id"),
    ("customers", customers, "customer_id"),
]:
    if df[key].isna().any():
        raise ValueError(f"{df_name}.{key}에 결측값이 있습니다.")

if not orders["order_id"].is_unique:
    raise ValueError("orders.order_id는 고유해야 합니다.")

if not customers["customer_id"].is_unique:
    raise ValueError("customers.customer_id는 고유해야 합니다.")

# orders.customer_id는 여러 주문에서 반복될 수 있으므로 고유성을 요구하지 않음

# 1. 주문 상품별 금액 계산
items = order_items.copy()
items["line_total"] = items["quantity"] * items["unit_price"]

# 2. 주문 상품(many)과 주문(one) 병합
rows_before = len(items)
items_with_orders = items.merge(
    orders[["order_id", "customer_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="_order_match",
)
rows_after_orders = len(items_with_orders)

unmatched_orders = (items_with_orders["_order_match"] == "left_only").sum()

print(
    f"order_items → orders: 병합 전 {rows_before:,}행, "
    f"병합 후 {rows_after_orders:,}행, 비매칭 {unmatched_orders:,}행"
)

if rows_before != rows_after_orders:
    raise ValueError("orders 병합 후 행 수가 달라졌습니다.")
if unmatched_orders:
    raise ValueError("orders에서 매칭되지 않은 order_id가 있습니다.")

# 3. 고객 ID가 customers에 존재하는지 확인
rows_before_customers = len(items_with_orders)
items_with_customers = items_with_orders.merge(
    customers[["customer_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_match",
)
rows_after_customers = len(items_with_customers)

unmatched_customers = (
    items_with_customers["_customer_match"] == "left_only"
).sum()

print(
    f"주문 데이터 → customers: 병합 전 {rows_before_customers:,}행, "
    f"병합 후 {rows_after_customers:,}행, 비매칭 {unmatched_customers:,}행"
)

if rows_before_customers != rows_after_customers:
    raise ValueError("customers 병합 후 행 수가 달라졌습니다.")
if unmatched_customers:
    raise ValueError("customers에서 매칭되지 않은 customer_id가 있습니다.")

# 4. completed 주문만 선택
completed_items = items_with_customers.loc[
    items_with_customers["order_status"].eq("completed")
].copy()

# completed 주문 상품 금액의 원본 합계
completed_total = completed_items["line_total"].sum()

# 5. 고객별 completed 구매 금액
customer_totals = (
    completed_items
    .groupby("customer_id", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "completed_total"})
)

# 구매 기록이 없는 고객도 포함하고 금액은 0으로 표시
customer_purchase = customers.merge(
    customer_totals,
    on="customer_id",
    how="left",
    validate="one_to_one",
)
customer_purchase["completed_total"] = (
    customer_purchase["completed_total"].fillna(0)
)

# 6. 원본 completed 합계와 고객별 합계 비교
customer_total_sum = customer_purchase["completed_total"].sum()

print(f"completed 원본 합계: {completed_total:,.2f}")
print(f"고객별 합계:         {customer_total_sum:,.2f}")

if not np.isclose(completed_total, customer_total_sum, rtol=1e-10, atol=1e-8):
    raise ValueError("completed 원본 합계와 고객별 합계가 일치하지 않습니다.")

# 결과 확인
customer_purchase

order_items → orders: 병합 전 764행, 병합 후 764행, 비매칭 0행
주문 데이터 → customers: 병합 전 764행, 병합 후 764행, 비매칭 0행
completed 원본 합계: 148,990,000.00
고객별 합계:         148,990,000.00


,customer_id,name,gender,age,city,signup_date,completed_total
0,1,김수민,F,19,광주,2024-08-25,0.0
1,2,김정호,F,32,대구,2026-01-08,0.0
2,3,이경수,F,61,성남,2024-08-18,3178000.0
3,4,조영호,F,55,울산,2026-06-19,603000.0
4,5,이예원,F,19,부산,2024-11-19,2004000.0
...,...,...,...,...,...,...,...
145,146,김숙자,M,61,성남,2026-02-28,1000000.0
146,147,이정남,M,19,부산,2025-04-19,2990000.0
147,148,오도현,M,29,고양,2026-08-21,271000.0
148,149,김정자,M,20,부산,2024-12-25,1295000.0


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [108]:
# 과제 코드를 아래에 작성하세요.

#1번
customers_over_40=customers[
    customers['age']>=40
]
print("40세 이상 최소연령:", customers_over_40['age'].min() )
print(customers_over_40.head())

#2번
print("2번: ", products.sort_values('price',ascending=False).head(10))

#3번
orders_payment=orders.groupby('payment_method', as_index=False).agg(total_order=('order_id','nunique')
).sort_values('total_order', ascending=False)
orders_payment["orders_ratio"]=(
    orders_payment['total_order']/orders_payment['total_order'].sum()*100
    ).round(2)
display(orders_payment)

40세 이상 최소연령: 40
   customer_id name gender  age city signup_date
2            3  이경수      F   61   성남  2024-08-18
3            4  조영호      F   55   울산  2026-06-19
6            7  이상현      F   53   인천  2025-02-17
8            9  송지민      M   69   서울  2025-12-25
9           10  유도현      F   62   울산  2024-12-12
2번:      product_id product_name category   price
98          99    뷰티 상품 099       뷰티  200000
69          70    패션 상품 070       패션  198000
57          58    식품 상품 058       식품  197000
42          43    뷰티 상품 043       뷰티  197000
23          24   스포츠 상품 024      스포츠  196000
8            9   스포츠 상품 009      스포츠  193000
36          37    뷰티 상품 037       뷰티  193000
71          72    뷰티 상품 072       뷰티  189000
7            8   스포츠 상품 008      스포츠  189000
52          53  생활용품 상품 053     생활용품  188000


,payment_method,total_order,orders_ratio
2,kakao_pay,79,26.33
3,naver_pay,77,25.67
0,bank_transfer,74,24.67
1,card,70,23.33


In [ ]:
#4번
category_average=products.groupby('category', as_index=False).agg(
    total_price=('price','sum'),
    total_products_number=('product_id', 'nunique'),
)
category_average['average_price']=(
    category_average['total_price']/category_average['total_products_number']
).round(2)

category_average=category_average
display(category_average.sort_values('average_price', ascending=False))

,category,total_price,total_products_number,average_price
4,식품,960000,7,137142.86
1,뷰티,1883000,16,117687.50
6,패션,1275000,11,115909.09
3,스포츠,2120000,19,111578.95
0,도서,1496000,14,106857.14
5,전자기기,1727000,17,101588.24
2,생활용품,1543000,16,96437.50


In [112]:
#5번
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

print("전체 금액 합계: ", order_sales['line_total'].sum())
print("completed 금액 합계: ", completed_order_sales['line_total'].sum())

orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

전체 금액 합계:  255610000
completed 금액 합계:  148990000


##### 6번
customers와 orders, order_items를 사용해 고객별 구매 금액을 계산하려고 함.
다음 조건을 포함해 python 코드를 짤 것.

1. quantity*unit_price로 line_total 생성
2. orders.order_id, customer_id의 고유성 확인
3. many_to_one merge와 indicator 사용
4. 병합 전후 행 수와 비매칭 확인
5. order_status==completed만 필터링
6. 원본 completed total과 고객별 total 합계를 비교

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 주문 건수가 전체 판매액을 결정한다.
2. 스포츠 카테고리가 가장 큰 판매액을 보였지만, 단일 품목만이 잘 팔리는 것이 아니다.

### 그 결과를 뒷받침하는 수치/표
- 월별 주문 건수는 최소 3건, 최대 24건, 평균 14.15건이었고, 평균 주문 금액 최소는 530,000, 최대는 1,116,000, 평균은 810,650.77 이었다. 

- 카테고리별 주문금액에서 스포츠 카테고리가 21%로 가장 많이 팔렸다. 또한, 상품별 주문금액에서 상위 10개 항목중 3개가 스포츠로 카테고리 중 가장 많았다.

### 추가로 확인하고 싶은 질문
추가적인 데이터가 존재한다면 시기별 마케팅 금액과 주문 건수 사이 상관관계가 존재할까?

### 현재 결과의 한계
판매액의 단위를 알 수 없고, 단순히 판매금액을 알 뿐 할인이 적용되었는지 여부를 알 수 없다.
또한, 월별 판매건수가 차이가 나는데 원인이 시기인지, 마케팅인지 등 현재의 데이터 외적인 내용은 알 수 없다.

## 최종 제출 체크
- [O] 핵심 셀 Output이 남아 있습니다.